In [3]:
import cv2
import mediapipe as mp
import numpy as np
import os
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report


# =========================================================
# 1. MEDIAPIPE SETUP
# =========================================================

BaseOptions = mp.tasks.BaseOptions
HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
RunningMode = mp.tasks.vision.RunningMode


# =========================================================
# 2. FUNCTION TO CONVERT ONE IMAGE INTO 63 FEATURES
# =========================================================

def get_landmarks(image, detector):

    # Convert BGR → RGB
    rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Convert to MediaPipe image
    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb
    )

    # Detect hand
    result = detector.detect(mp_image)

    # No hand found
    if not result.hand_landmarks:
        return None

    # Take first hand
    landmarks = result.hand_landmarks[0]

    # Wrist coordinates
    wrist_x = landmarks[0].x
    wrist_y = landmarks[0].y
    wrist_z = landmarks[0].z

    features = []

    # Store x, y, z for all 21 landmarks
    for landmark in landmarks:

        x = landmark.x - wrist_x
        y = landmark.y - wrist_y
        z = landmark.z - wrist_z

        features.extend([x, y, z])

    return features


# =========================================================
# 3. CREATE MEDIAPIPE HAND DETECTOR FOR IMAGES
# =========================================================

image_options = HandLandmarkerOptions(
    base_options=BaseOptions(
        model_asset_path="hand_landmarker.task"
    ),
    running_mode=RunningMode.IMAGE,
    num_hands=1
)


# =========================================================
# 4. READ DATASET
# =========================================================

X = []
y = []

dataset_path = "dataset"

print("Reading dataset...")


with HandLandmarker.create_from_options(image_options) as detector:

    # Go through A, B, C ... Z folders
    for letter in sorted(os.listdir(dataset_path)):

        letter_path = os.path.join(dataset_path, letter)

        # Ignore files
        if not os.path.isdir(letter_path):
            continue

        print("\nProcessing:", letter)

        # Count images
        total_images = 0
        usable_images = 0

        # Go through images in that letter's folder
        for filename in os.listdir(letter_path):

            image_path = os.path.join(letter_path, filename)

            image = cv2.imread(image_path)

            if image is None:
                continue

            total_images += 1

            # Extract 21 landmarks
            features = get_landmarks(image, detector)

            # If a hand was detected
            if features is not None:

                X.append(features)
                y.append(letter)

                usable_images += 1

        print(
            f"{letter}: {total_images} images → "
            f"{usable_images} usable"
        )


print("\nDataset processing complete.")

print("Total usable images:", len(X))


# =========================================================
# 5. CONVERT TO NUMPY ARRAY
# =========================================================

X = np.array(X)
y = np.array(y)

print("Feature shape:", X.shape)


# =========================================================
# 6. SPLIT INTO TRAINING AND TESTING DATA
# =========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining images:", len(X_train))
print("Testing images:", len(X_test))


# =========================================================
# 7. CREATE RANDOM FOREST
# =========================================================

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)


# =========================================================
# 8. TRAIN RANDOM FOREST
# =========================================================

print("\nTraining Random Forest...")

model.fit(X_train, y_train)

print("Training complete!")


# =========================================================
# 9. TEST THE MODEL
# =========================================================

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("\nTest Accuracy:", accuracy * 100, "%")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


# =========================================================
# 10. SAVE THE TRAINED MODEL
# =========================================================

joblib.dump(model, "asl_random_forest.pkl")

print("\nModel saved as asl_random_forest.pkl")


# =========================================================
# 11. LIVE CAMERA DETECTION
# =========================================================

live_options = HandLandmarkerOptions(
    base_options=BaseOptions(
        model_asset_path="hand_landmarker.task"
    ),
    running_mode=RunningMode.VIDEO,
    num_hands=1
)


with HandLandmarker.create_from_options(live_options) as detector:

    cap = cv2.VideoCapture(0)

    timestamp = 0

    while True:

        success, frame = cap.read()

        if not success:
            break

        # Mirror camera
        frame = cv2.flip(frame, 1)

        # Convert BGR → RGB
        rgb = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        # MediaPipe image
        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=rgb
        )

        # Detect hand
        result = detector.detect_for_video(
            mp_image,
            timestamp
        )

        timestamp += 1

        # If hand detected
        if result.hand_landmarks:

            landmarks = result.hand_landmarks[0]

            # Wrist
            wrist_x = landmarks[0].x
            wrist_y = landmarks[0].y
            wrist_z = landmarks[0].z

            features = []

            # Create same 63 features
            for landmark in landmarks:

                x = landmark.x - wrist_x
                y = landmark.y - wrist_y
                z = landmark.z - wrist_z

                features.extend([x, y, z])

            # Convert to numpy
            features = np.array(features).reshape(1, -1)

            # Predict letter
            prediction = model.predict(features)[0]

            # Get confidence
            probabilities = model.predict_proba(features)

            confidence = np.max(probabilities) * 100

            # Display prediction
            cv2.putText(
                frame,
                "Letter: " + prediction,
                (30, 60),
                cv2.FONT_HERSHEY_SIMPLEX,
                1.5,
                (0, 255, 0),
                3
            )

            # Display confidence
            cv2.putText(
                frame,
                f"Confidence: {confidence:.1f}%",
                (30, 100),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (255, 255, 255),
                2
            )

            # Draw landmarks
            for landmark in landmarks:

                x = int(
                    landmark.x * frame.shape[1]
                )

                y = int(
                    landmark.y * frame.shape[0]
                )

                cv2.circle(
                    frame,
                    (x, y),
                    5,
                    (0, 255, 0),
                    -1
                )

        else:

            cv2.putText(
                frame,
                "No hand detected",
                (30, 60),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (0, 0, 255),
                2
            )

        # Show camera
        cv2.imshow(
            "ASL Gesture Translator",
            frame
        )

        # Press q to quit
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()

Reading dataset...

Processing: A
A: 101 images → 101 usable

Processing: B
B: 101 images → 101 usable

Processing: C
C: 102 images → 102 usable

Processing: D
D: 102 images → 102 usable

Processing: E
E: 101 images → 101 usable

Processing: F
F: 102 images → 102 usable

Processing: G
G: 101 images → 101 usable

Processing: H
H: 102 images → 99 usable

Processing: I
I: 100 images → 100 usable

Processing: K
K: 101 images → 101 usable

Processing: L
L: 100 images → 100 usable

Processing: M
M: 102 images → 96 usable

Processing: N
N: 100 images → 98 usable

Processing: O
O: 100 images → 100 usable

Processing: P
P: 102 images → 99 usable

Processing: Q
Q: 101 images → 101 usable

Processing: R
R: 100 images → 97 usable

Processing: S
S: 99 images → 99 usable

Processing: T
T: 100 images → 100 usable

Processing: U
U: 106 images → 97 usable

Processing: V
V: 102 images → 96 usable

Processing: W
W: 100 images → 98 usable

Processing: X
X: 102 images → 102 usable

Processing: Y
Y: 100 ima